# D1.6 · Distinguishing agent from human

**Function D — Security Operations → The SOC Analyst & Detection Engineer**  ·  *Security of AI*

---

**Risk.** Your earliest Shadow Autonomy signal is invisible.

**Control.** Behavioural signatures separating agent from inherited human.

**This lab.** Tell an agent apart from the human whose credential it inherited.

| | |
|---|---|
| Open-source tooling | OpenSearch |
| Open-weight models | Llama 3.3 |

> Runs anywhere: standard library only, no network, no API key. Where a lesson names a real tool you would deploy (Falco, OPA, SPIRE, Keycloak), the notebook models the *decision* that tool makes, so the lesson still lands on a machine that cannot pull containers.

In [ ]:
# --- Cyber Commons bootstrap -------------------------------------------------
# Puts the lab library on the path. Works from a clone, from the repo root, and
# on Kaggle. Standard library only — nothing to install, no network required.
import sys, os, subprocess
from pathlib import Path

def _find_labs():
    for base in [Path.cwd(), *Path.cwd().parents]:
        if (base / "labs" / "cybercommons" / "__init__.py").is_file():
            return base / "labs"
    # Kaggle kernels start in /kaggle/working with the repo absent. If the
    # kernel has internet enabled we clone it; if not, this raises and the
    # message tells you to attach the repo as a dataset instead.
    dest = Path("/kaggle/working/cyber-commons")
    if not dest.exists():
        subprocess.run(["git", "clone", "--depth", "1", "--branch", "claude/vulnbench-setup-scheduling-81aqov",
                        "https://github.com/spbreed/cyber-commons", str(dest)], check=True)
    return dest / "labs"

sys.path.insert(0, str(_find_labs()))
import cybercommons
print(cybercommons.banner("D1.6"))

Distinguishing agent from human in telemetry, without a registry — because the ones you most need to find are the ones not in it.

In [ ]:
from cybercommons import soc
import time

now = time.time()
events  = [soc.Event(now + i * 0.05, "svc-indexer", "read_file") for i in range(120)]
events += [soc.Event(now + t, "dana", "read_file")
           for t in (0, 4, 11, 12, 60, 130, 133, 400, 900)]
events += [soc.Event(now + i * 1.2, "shared-account", "http_get") for i in range(30)]

for actor in ("svc-indexer", "dana", "shared-account"):
    r = soc.agent_score(events, actor)
    print(f"{actor:16s} score={r['score']:.3f} {r['verdict']:8s} {r['signals']}")

Now the honest part: where the heuristic is wrong.

In [ ]:
# a human using an IDE with autosave looks metronomic
ide = [soc.Event(now + i * 2.0, "sam", "write_file") for i in range(40)]
print(soc.agent_score(ide, "sam"), "  ← a person, scored as software")

# an agent with human-paced backoff looks human
polite = [soc.Event(now + t, "slow-agent", "http_get")
          for t in (0, 7, 19, 44, 90, 210, 480)]
print(soc.agent_score(polite, "slow-agent"), "  ← software, scored as a person")

### Expect

The service and shared accounts score as agents while `dana` scores human. The two adversarial cases misclassify in both directions.

### Your turn

Both errors matter differently: a misclassified human triggers an investigation, a misclassified agent hides. Which error would you tune toward, and what does that say about the threshold?

---

[All lessons](https://github.com/spbreed/cyber-commons/tree/claude/vulnbench-setup-scheduling-81aqov/labs/notebooks) · [Lesson page](https://spbreed.github.io/cyber-commons/lessons/D1.6.html) · [Lab library](https://github.com/spbreed/cyber-commons/tree/claude/vulnbench-setup-scheduling-81aqov/labs/cybercommons)

*Cyber Commons — a free, open commons for Cyber AI.*